In [ ]:
!date

### hypothesis: leiden clusters called independently in each assay are concordant with each other

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

import anndata
import scanpy as sc

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
tmp_order = ['Start', 'Sendai', 'Fail1', 'Delayed', 'Inter1', 'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/integrate_m3c/merged.processed.harmony.h5ad')

In [ ]:
adata

In [ ]:
adata.obs.head()

In [ ]:
adata.obs.index = [x[:-2] for x in adata.obs.index]

In [ ]:
sc.pl.umap(adata, color=['time', 'line', 'assay'], ncols=2)

### load in leiden clusters 

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc'

mct_leidens = []
for s in donors:
    tmp_leiden = pd.read_csv(f'{indir}/csv/leiden/per_donor/{s}.chrom100k.csv', sep='\t', index_col=0)
    mct_leidens.append(tmp_leiden)

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snm3C/mc'

m3c_leidens = []
for s in donors:
    tmp_leiden = pd.read_csv(f'{indir}/csv/leiden/{s}.chrom100k.csv', sep='\t', index_col=0)
    tmp_leiden.index = [x.replace('_indexed', '') for x in tmp_leiden.index]
    m3c_leidens.append(tmp_leiden)

In [ ]:
merged_mct = pd.concat(mct_leidens)
merged_m3c = pd.concat(m3c_leidens)
adata.obs['m3c_leiden'] = merged_m3c.reindex(adata.obs.index)['leiden']
adata.obs['mct_leiden'] = merged_mct.reindex(adata.obs.index)['leiden']

In [ ]:
mct_adata = adata[~adata.obs['mct_leiden'].isna()]
m3c_adata = adata[~adata.obs['m3c_leiden'].isna()]

mct_adata.obs['mct_leiden'] = [str(int(x)) for x in mct_adata.obs['mct_leiden']]
# mct_adata.obs['mct_leiden'] = mct_adata.obs['mct_leiden'].astype(str)

m3c_adata.obs['m3c_leiden'] = [str(int(x)) for x in m3c_adata.obs['m3c_leiden']]
# m3c_adata.obs['m3c_leiden'] = m3c_adata.obs['m3c_leiden'].astype(str)

In [ ]:
sc.tl.leiden(adata, resolution=0.4)

In [ ]:
sc.pl.umap(adata, color=['leiden'], ncols=2)

In [ ]:
sc.pl.umap(adata, color=['mct_leiden', 'm3c_leiden'], ncols=2)

In [ ]:
# sc.pl.umap(mct_adata, color=['mct_leiden'], ncols=2)

In [ ]:
sc.pl.umap(m3c_adata, color=['m3c_leiden'], ncols=2)

In [ ]:
!date